In [1]:
%load_ext autoreload
%autoreload 2
%cd /home/albin/egna_proj/block_puzzle_rl/

/home/albin/egna_proj/block_puzzle_rl


/home/albin/egna_proj/block_puzzle_rl/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
# ⬇️ SB3 + Gym imports
import numpy as np
import torch
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from gymnasium import spaces

# ⬇️ Your imports
from game.block_puzzle_env import BlockPuzzleEnv
from agent.utils import encode_state, encode_state_cnn
from game.block import random_block

In [3]:

# ⬇️ Custom wrapper to convert dict obs → flat array
class FlattenedBlockEnv(gym.Env):
    def __init__(self, width=12, height=12, num_blocks=3):
        super().__init__()
        self.raw_env = BlockPuzzleEnv(width, height, num_blocks)
        self.action_space = self.raw_env.action_space
        dummy_obs, _ = self.raw_env.reset()
        sample_obs = encode_state_cnn(dummy_obs)
        self.observation_space = gym.spaces.Box(
            low=0, high=255, shape=sample_obs.shape, dtype=np.uint8
        )
    
    def reset(self, seed=None, options=None):
        obs_dict, _ = self.raw_env.reset()
        flat_obs = encode_state_cnn(obs_dict)
        return flat_obs.astype(np.uint8), {}

    def step(self, action):
        # Handle batched action from DummyVecEnv (e.g., [0, 9, 10])
        if isinstance(action, (list, np.ndarray)) and len(action) == 3:
            a0, a1, a2 = map(int, action)
        else:
            # Flat index → unravel into (block_index, row, col)
            a0, a1, a2 = np.unravel_index(action, self.action_space.nvec)

        raw_action = (a0, a1, a2)
        (obs_dict, _), reward, terminated, truncated, info = self.raw_env.step(raw_action)
        flat_obs = encode_state_cnn(obs_dict)
        return flat_obs.astype(np.float32), reward, terminated, truncated, info


    def render(self):
        return self.raw_env.render()

    def close(self):
        return self.raw_env.close()


class DiscreteActionWrapper(gym.Env):
    def __init__(self, raw_env):
        super().__init__()
        self.raw_env = raw_env
        self.original_action_space = raw_env.action_space  # Should be MultiDiscrete
        self.obs_space = self.raw_env.observation_space

        # Flatten MultiDiscrete([a, b, c]) → Discrete(a * b * c)
        self.action_space = spaces.Discrete(np.prod(self.original_action_space.nvec))
        self.observation_space = spaces.Box(low=0, high=255, shape=encode_state_cnn(self.raw_env.reset()[0]).shape, dtype=np.uint8)

    def reset(self, **kwargs):
        obs_dict, _ = self.raw_env.reset(**kwargs)
        return encode_state_cnn(obs_dict).astype(np.uint8), {}

    def step(self, flat_action):
        a0, a1, a2 = np.unravel_index(flat_action, self.original_action_space.nvec)
        action = (int(a0), int(a1), int(a2))
        (obs_dict, _), reward, terminated, truncated, info = self.raw_env.step(action)
        flat_obs = encode_state_cnn(obs_dict)
        return flat_obs.astype(np.uint8), reward, terminated, truncated, info

    def render(self, **kwargs):
        return self.raw_env.render(**kwargs)



In [4]:
raw_env = BlockPuzzleEnv(width=12, height=12, num_blocks=3, block_function=random_block)
wrapped_env = DiscreteActionWrapper(raw_env)
monitored_env = Monitor(wrapped_env)

check_env(wrapped_env, warn=True)  # ✅ Now this should pass
vec_env = DummyVecEnv([lambda: monitored_env])

model = PPO(
    policy="CnnPolicy",  # or "CnnPolicy" if your input is image-like
    env=vec_env,
    learning_rate=1e-3,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    verbose=1,
    tensorboard_log="./sb3_logs/", 
    
)


model.learn(total_timesteps=100_000)
model.save("sb3_block_ppo_cnn")


Using cuda device
Logging to ./sb3_logs/PPO_4
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 2.26     |
|    ep_rew_mean     | -0.61    |
| time/              |          |
|    fps             | 288      |
|    iterations      | 1        |
|    time_elapsed    | 7        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 2.61        |
|    ep_rew_mean          | -0.575      |
| time/                   |             |
|    fps                  | 209         |
|    iterations           | 2           |
|    time_elapsed         | 19          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.021895042 |
|    clip_fraction        | 0.443       |
|    clip_range           | 0.2         |
|    entropy_loss         | -6.05       |
|    explained_variance   

KeyboardInterrupt: 

In [ ]:
obs, _ = wrapped_env.reset()
done = False
total_reward = 0

while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = wrapped_env.step(action)
    total_reward += reward
    done = terminated or truncated
    wrapped_env.render()
    print()

print(f"Total reward (greedy run): {total_reward}")

□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ ■ □ □ □ □ □
□ □ □ □ □ □ ■ ■ □ □ □ □
□ □ □ □ □ □ □ ■ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □

□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ ■ □ □ □ □ □
■ □ □ □ □ □ ■ ■ □ □ □ □
□ □ □ □ □ □ □ ■ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □

□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ ■ □ □ □ □ □
■ □ □ □ □ □ ■ ■ □ □ □ □
□ □ □ □ □ □ □ ■ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
■ ■ □ □ □ □ □ □ □ □ □ □
□ ■ ■ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □

■ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ □ □ □ □ □ □
□ □ □ □ □ □ ■ □ □ □ □ □
■ □ □ □ □ □ ■